In [ ]:
"""
Script 7: Hypothesis Testing
Perform t-tests, paired t-tests, ANOVA, and other statistical tests
"""

import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
from pathlib import Path

def one_sample_t_test(data, column_name, test_value=0):
    """Perform one-sample t-test"""
    data_clean = data.dropna()
    
    t_statistic, p_value = scipy_stats.ttest_1samp(data_clean, test_value)
    
    result = {
        'Test': 'One-Sample t-test',
        'Column': column_name,
        'Test Value': test_value,
        'Sample Mean': data_clean.mean(),
        't-statistic': t_statistic,
        'P-value': p_value,
        'Significant (α=0.05)': 'Yes' if p_value < 0.05 else 'No',
        'Conclusion': f'Mean significantly {"≠" if p_value < 0.05 else "="} {test_value}'
    }
    
    return result

def two_sample_t_test(data1, data2, name1, name2):
    """Perform independent two-sample t-test"""
    d1 = data1.dropna()
    d2 = data2.dropna()
    
    # Check for equal variances using Levene's test
    levene_stat, levene_p = scipy_stats.levene(d1, d2)
    equal_var = levene_p > 0.05
    
    # Perform t-test
    t_statistic, p_value = scipy_stats.ttest_ind(d1, d2, equal_var=equal_var)
    
    result = {
        'Test': 'Two-Sample t-test',
        'Group 1': name1,
        'Group 2': name2,
        'Mean 1': d1.mean(),
        'Mean 2': d2.mean(),
        'Mean Difference': d1.mean() - d2.mean(),
        't-statistic': t_statistic,
        'P-value': p_value,
        'Equal Variance': 'Yes' if equal_var else 'No',
        'Significant (α=0.05)': 'Yes' if p_value < 0.05 else 'No',
        'Conclusion': 'Means are significantly different' if p_value < 0.05 else 'No significant difference'
    }
    
    return result

def paired_t_test(data1, data2, name1, name2):
    """Perform paired t-test"""
    # Get common indices
    common_idx = data1.index.intersection(data2.index)
    d1 = data1.loc[common_idx].dropna()
    d2 = data2.loc[common_idx].dropna()
    
    # Ensure same length
    min_len = min(len(d1), len(d2))
    d1 = d1.iloc[:min_len]
    d2 = d2.iloc[:min_len]
    
    t_statistic, p_value = scipy_stats.ttest_rel(d1, d2)
    
    result = {
        'Test': 'Paired t-test',
        'Group 1': name1,
        'Group 2': name2,
        'Mean 1': d1.mean(),
        'Mean 2': d2.mean(),
        'Mean Difference': d1.mean() - d2.mean(),
        't-statistic': t_statistic,
        'P-value': p_value,
        'Sample Size': len(d1),
        'Significant (α=0.05)': 'Yes' if p_value < 0.05 else 'No',
        'Conclusion': 'Means are significantly different' if p_value < 0.05 else 'No significant difference'
    }
    
    return result

def mann_whitney_u_test(data1, data2, name1, name2):
    """Perform Mann-Whitney U test (non-parametric alternative to t-test)"""
    d1 = data1.dropna()
    d2 = data2.dropna()
    
    u_statistic, p_value = scipy_stats.mannwhitneyu(d1, d2, alternative='two-sided')
    
    result = {
        'Test': 'Mann-Whitney U test',
        'Group 1': name1,
        'Group 2': name2,
        'Median 1': d1.median(),
        'Median 2': d2.median(),
        'U-statistic': u_statistic,
        'P-value': p_value,
        'Significant (α=0.05)': 'Yes' if p_value < 0.05 else 'No',
        'Conclusion': 'Distributions are significantly different' if p_value < 0.05 else 'No significant difference'
    }
    
    return result

def print_test_result(result):
    """Print test result"""
    print(f"\n{result['Test']}")
    print("─" * 80)
    for key, value in result.items():
        if key != 'Test':
            if isinstance(value, float):
                print(f"  {key:.<40} {value:>15.6f}")
            else:
                print(f"  {key:.<40} {str(value):>15}")

def hypothesis_testing(df):
    """Perform various hypothesis tests"""
    print(f"\n{'='*80}")
    print("HYPOTHESIS TESTING")
    print(f"{'='*80}")
    
    all_results = []
    
    # 1. One-sample t-test: Is mean significantly different from population mean?
    print(f"\n{'='*80}")
    print("1. ONE-SAMPLE T-TEST")
    print(f"{'='*80}")
    print("H0: Mean Close Price = median of Close Price")
    print("Ha: Mean Close Price ≠ median of Close Price")
    
    median_price = df['Close Price'].median()
    result_1 = one_sample_t_test(df['Close Price'], 'Close Price', median_price)
    print_test_result(result_1)
    all_results.append(result_1)
    
    # 2. Paired t-test: Compare 3-Day and 5-Day moving averages
    print(f"\n{'='*80}")
    print("2. PAIRED T-TEST (Moving Averages Comparison)")
    print(f"{'='*80}")
    print("H0: Mean of 3-Day MA = Mean of 5-Day MA")
    print("Ha: Mean of 3-Day MA ≠ Mean of 5-Day MA")
    
    result_2 = paired_t_test(df['3 Day MV.'], df['5 Day MV.'], '3-Day MA', '5-Day MA')
    print_test_result(result_2)
    all_results.append(result_2)
    
    # 3. Split data in half and perform two-sample t-test
    print(f"\n{'='*80}")
    print("3. TWO-SAMPLE T-TEST (First Half vs Second Half)")
    print(f"{'='*80}")
    print("H0: Mean Close Price (1st half) = Mean Close Price (2nd half)")
    print("Ha: Mean Close Price (1st half) ≠ Mean Close Price (2nd half)")
    
    mid_point = len(df) // 2
    result_3 = two_sample_t_test(df['Close Price'].iloc[:mid_point], 
                                 df['Close Price'].iloc[mid_point:],
                                 'First Half', 'Second Half')
    print_test_result(result_3)
    all_results.append(result_3)
    
    # 4. Mann-Whitney U test on split data
    print(f"\n{'='*80}")
    print("4. MANN-WHITNEY U TEST (Non-parametric)")
    print(f"{'='*80}")
    print("H0: Distributions of first and second half are equal")
    print("Ha: Distributions are different")
    
    result_4 = mann_whitney_u_test(df['Close Price'].iloc[:mid_point],
                                   df['Close Price'].iloc[mid_point:],
                                   'First Half', 'Second Half')
    print_test_result(result_4)
    all_results.append(result_4)
    
    # 5. Levene's test for equality of variances
    print(f"\n{'='*80}")
    print("5. LEVENE'S TEST (Equality of Variances)")
    print(f"{'='*80}")
    print("H0: Variances are equal")
    print("Ha: Variances are not equal")
    
    levene_stat, levene_p = scipy_stats.levene(df['Close Price'].iloc[:mid_point],
                                               df['Close Price'].iloc[mid_point:])
    
    levene_result = {
        'Test': "Levene's Test",
        'Statistic': levene_stat,
        'P-value': levene_p,
        'Significant (α=0.05)': 'Yes' if levene_p < 0.05 else 'No',
        'Conclusion': 'Variances are significantly different' if levene_p < 0.05 else 'Variances are equal'
    }
    print_test_result(levene_result)
    all_results.append(levene_result)
    
    return pd.DataFrame(all_results)

def effect_size_analysis(df):
    """Calculate effect sizes"""
    print(f"\n{'='*80}")
    print("EFFECT SIZE ANALYSIS")
    print(f"{'='*80}\n")
    
    # Cohen's d for first half vs second half
    mid_point = len(df) // 2
    d1 = df['Close Price'].iloc[:mid_point].dropna()
    d2 = df['Close Price'].iloc[mid_point:].dropna()
    
    # Calculate Cohen's d
    pooled_std = np.sqrt(((len(d1)-1)*d1.std()**2 + (len(d2)-1)*d2.std()**2) / (len(d1) + len(d2) - 2))
    cohens_d = (d1.mean() - d2.mean()) / pooled_std
    
    print(f"Cohen's d (First Half vs Second Half): {cohens_d:.4f}")
    
    if abs(cohens_d) < 0.2:
        interpretation = "Negligible effect"
    elif abs(cohens_d) < 0.5:
        interpretation = "Small effect"
    elif abs(cohens_d) < 0.8:
        interpretation = "Medium effect"
    else:
        interpretation = "Large effect"
    
    print(f"Interpretation: {interpretation}\n")
    
    return cohens_d

if __name__ == "__main__":
    file_path = 'outputs/cleaned_data.csv'
    
    try:
        df = pd.read_csv(file_path)
        df['Date'] = pd.to_datetime(df['Date'])
        
        # Perform hypothesis tests
        test_results = hypothesis_testing(df)
        
        # Effect size analysis
        effect_size_analysis(df)
        
        # Save results
        Path('outputs').mkdir(exist_ok=True)
        test_results.to_csv('outputs/hypothesis_tests.csv', index=False)
        
        print(f"\n✓ Hypothesis test results saved to outputs/hypothesis_tests.csv")
        print("✓ Hypothesis testing completed!")
        
    except FileNotFoundError:
        print(f"Please run 01_data_loading.py first to generate {file_path}")